# 04 – Sensitivity Studies

This notebook estimates the **statistical sensitivity** to the transverse tau polarization **P_T(τ)** as a function of integrated luminosity, and evaluates the expected systematic uncertainties.

**Goals:**
- Estimate signal yield after BDT selection
- Propagate statistical uncertainty to P_T(τ)
- Project sensitivity as a function of Belle II luminosity
- Identify dominant systematic uncertainties

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import uproot

try:
    import mplhep as hep
    hep.style.use(hep.style.Belle2)
except ImportError:
    pass

plt.rcParams['figure.dpi'] = 120

## 1. Physics Parameters

In [ ]:
# SM prediction for transverse tau polarization
PT_SM = 0.0    # update with the actual SM value from theory

# B → D(*) τ ν branching fraction (PDG)
BR_signal = 1.4e-2   # example for B → D* τ ν

# Υ(4S) → BB̄ production cross-section at Belle II [pb]
sigma_BB = 1.1e3   # ~1.1 nb

# FEI tagging efficiency × signal reconstruction efficiency (approximate)
eff_total = 0.01   # update after full selection study

# BDT working point signal efficiency (from notebook 03)
eff_bdt = 0.80     # update from 03_bdt.ipynb

print(f"Combined efficiency (selection × BDT): {eff_total * eff_bdt:.4f}")

## 2. Expected Signal Yield vs. Luminosity

In [ ]:
# Luminosities to scan [fb^-1]
luminosities = np.array([1, 5, 10, 50, 100, 250, 500, 1000])  # fb^-1

# Number of BB̄ pairs (two B per event, factor 2)
N_B = 2 * sigma_BB * luminosities * 1e3   # pb^-1 × pb = dimensionless

# Signal yield after all selections
N_sig = N_B * BR_signal * eff_total * eff_bdt

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(luminosities, N_sig, marker="o", color="steelblue")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"Integrated luminosity [fb$^{-1}$]")
ax.set_ylabel("Expected signal yield")
ax.set_title("Signal yield vs. luminosity")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

df_yield = pd.DataFrame({"L [fb-1]": luminosities, "N_sig": N_sig.astype(int)})
print(df_yield.to_string(index=False))

## 3. Statistical Uncertainty on P_T(τ)

In [ ]:
# Analysing power A: sensitivity of the angular observable to P_T
# For τ → π ν: A ≈ 1.  Update with the actual value for your observable.
A = 1.0

# Statistical uncertainty on P_T from N_sig events
# sigma(P_T) = 1 / (A * sqrt(N_sig))
sigma_PT = 1.0 / (A * np.sqrt(np.maximum(N_sig, 1)))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(luminosities, sigma_PT, marker="o", color="tomato", label=r"$\sigma(P_T)$ (stat.)")
ax.axhline(0.05, color="gray", linestyle="--", label="5% target")
ax.set_xscale("log")
ax.set_xlabel(r"Integrated luminosity [fb$^{-1}$]")
ax.set_ylabel(r"$\sigma(P_T^{\tau})$")
ax.set_title(r"Statistical sensitivity to $P_T(\tau)$")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig("../docs/sensitivity_vs_luminosity.pdf")
plt.show()

df_sens = pd.DataFrame({"L [fb-1]": luminosities, "sigma(P_T)": sigma_PT.round(4)})
print(df_sens.to_string(index=False))

## 4. Systematic Uncertainties

*(Fill in after performing detailed studies.)*

| Source | Estimated δP_T | Comment |
|---|---|---|
| FEI tagging efficiency | ... | |
| Reconstruction efficiency (π, D) | ... | |
| MC generator model | ... | |
| Form factor uncertainties | ... | |
| Background subtraction | ... | |
| **Total systematic** | ... | |

## 5. Luminosity Required for 5σ Observation

In [ ]:
# For a given hypothetical P_T value, estimate luminosity for 5σ observation
PT_hyp   = 0.10   # hypothetical true P_T
n_sigma  = 5.0    # significance target

# N_sig needed: A * P_T * sqrt(N) = n_sigma  =>  N = (n_sigma / (A * P_T))^2
N_needed = (n_sigma / (A * abs(PT_hyp)))**2
L_needed = N_needed / (2 * sigma_BB * 1e3 * BR_signal * eff_total * eff_bdt)

print(f"For P_T = {PT_hyp:.2f}:")
print(f"  Required signal yield : {N_needed:.0f}")
print(f"  Required luminosity   : {L_needed:.1f} fb^-1")

## 6. Conclusions

*(Fill in your conclusions after running the notebook on real data.)*

- At the current Belle II target luminosity of 50 ab⁻¹, the expected sensitivity is ...
- The dominant limiting factor is ...
- Proposed improvements: ...